
# WAKIS FD validation: round resistive-wall beam pipe

This notebook sets up a **circular beam pipe with a conductive wall** and compares the
computed **longitudinal beam impedance** against the analytical thick-wall resistive-wall model.

Validation targets:
- frequency scaling: $Z_\parallel \propto \sqrt{f}$
- conductivity scaling: $Z_\parallel \propto 1/\sqrt{\sigma}$
- approximate agreement in absolute magnitude

**Important notes**
- The example uses the current **FD impedance sweep** with **SIBC**.
- The geometry is a **finite pipe section**, so small deviations from the ideal infinite-pipe formula are expected.
- The notebook assumes that the longitudinal open-boundary keyword is `"ABC"` in your local WAKIS build. If your build uses a different keyword, adjust `bc_low` / `bc_high` accordingly.
- We use `conformal_pec_snap_fraction = 0.2`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.constants import mu_0, pi

import trimesh

from wakis import GridFIT3D, SolverFIT3D_FD, WakeSolver, FieldMonitor


## 1. Geometry helper: hollow circular pipe as STL

We generate a **watertight hollow cylinder** (tube) with open bore and annular end caps.
This is convenient for the conformal STL pipeline and allows a centered beam to pass through the pipe.


In [ ]:
def make_hollow_cylinder_stl(path, r_inner, r_outer, length, nphi=96):
    # Create a watertight hollow cylinder aligned with z and save as STL.
    phi = np.linspace(0.0, 2.0 * np.pi, nphi, endpoint=False)
    z0 = -0.5 * length
    z1 = 0.5 * length

    outer0 = np.column_stack(
        [r_outer * np.cos(phi), r_outer * np.sin(phi), np.full_like(phi, z0)]
    )
    outer1 = np.column_stack(
        [r_outer * np.cos(phi), r_outer * np.sin(phi), np.full_like(phi, z1)]
    )
    inner0 = np.column_stack(
        [r_inner * np.cos(phi), r_inner * np.sin(phi), np.full_like(phi, z0)]
    )
    inner1 = np.column_stack(
        [r_inner * np.cos(phi), r_inner * np.sin(phi), np.full_like(phi, z1)]
    )

    vertices = np.vstack([outer0, outer1, inner0, inner1])
    o0 = 0
    o1 = nphi
    i0 = 2 * nphi
    i1 = 3 * nphi

    faces = []
    for p in range(nphi):
        q = (p + 1) % nphi

        # outer cylindrical surface (outward normals)
        faces.append([o0 + p, o0 + q, o1 + q])
        faces.append([o0 + p, o1 + q, o1 + p])

        # inner cylindrical surface (reverse orientation: normals toward bore)
        faces.append([i0 + p, i1 + q, i0 + q])
        faces.append([i0 + p, i1 + p, i1 + q])

        # lower annulus z=z0
        faces.append([o0 + p, i0 + q, o0 + q])
        faces.append([o0 + p, i0 + p, i0 + q])

        # upper annulus z=z1
        faces.append([o1 + p, o1 + q, i1 + q])
        faces.append([o1 + p, i1 + q, i1 + p])

    mesh = trimesh.Trimesh(vertices=vertices, faces=np.asarray(faces), process=True)
    mesh.export(path)
    return mesh


## 2. Test setup

We choose a **centered round pipe** and compare the FD impedance to the analytical thick-wall result

\[
\frac{Z_\parallel(\omega)}{L} \approx \frac{Z_s(\omega)}{2\pi b},
\qquad
Z_s(\omega) = (1+j)\sqrt{\frac{\omega \mu_0}{2\sigma}}.
\]

for an on-axis beam in a circular pipe of radius $b$.

Depending on the exact Fourier convention used in your post-processing, the sign of the imaginary part
may appear with opposite sign. The real part should be unambiguous.


In [ ]:
# Pipe and material parameters
radius_inner = 10e-3  # 10 mm bore radius
wall_thickness = 2.0e-3  # 2 mm wall thickness
radius_outer = radius_inner + wall_thickness
pipe_length = 0.50  # 0.5 m conductive section
sigma_wall = 5.8e7  # copper-like conductivity
mu_r_wall = 1.0

# Computational box
xmin, xmax = -18e-3, 18e-3
ymin, ymax = -18e-3, 18e-3
zmin, zmax = -0.26, 0.26

# Grid (moderate first validation)
Nx, Ny, Nz = 52, 52, 220

# FD sweep and monitor
frequencies = np.linspace(5e6, 3.0e9, 200)
monitor_frequency = 1.0e9
eta = 0.2

# STL path
stl_path = Path("round_resistive_pipe_shell.stl")
mesh = make_hollow_cylinder_stl(
    stl_path,
    r_inner=radius_inner,
    r_outer=radius_outer,
    length=pipe_length,
    nphi=128,
)

mesh


## 3. Build FD solver

The pipe wall is modeled as an STL solid of material type `"sibc"`.
The user-facing impedance API still uses a `WakeSolver` object for beam parameters (`beta`, source/test offsets, etc.).
For FD we do **not** need `sigmaz` physically, so we leave it at its default and only set `beta=1.0`.


In [ ]:
def make_fd_solver(sigma=sigma_wall, verbose=0):
    grid = GridFIT3D(
        xmin=xmin,
        xmax=xmax,
        ymin=ymin,
        ymax=ymax,
        zmin=zmin,
        zmax=zmax,
        Nx=Nx,
        Ny=Ny,
        Nz=Nz,
        stl_solids={"pipe": str(stl_path)},
        stl_materials={"pipe": [1.0, mu_r_wall, sigma]},
        stl_material_types={"pipe": "sibc"},
        geometry_mode="conformal",
        stl_method="interior_points",
        bg_material_type="normal",
        conformal_pec_snap_fraction=eta,
        verbose=verbose,
    )

    solver = SolverFIT3D_FD(
        grid=grid,
        use_stl=True,
        use_sibc=False,  # legacy TD SIBC flag remains off
        bc_low=["PEC", "PEC", "ABC"],
        bc_high=["PEC", "PEC", "ABC"],
        verbose=verbose,
    )
    return solver


solver = make_fd_solver(verbose=0)
print("Active E DOFs:", len(solver.free_E_idx))
print("SIBC magnetic DOFs:", np.count_nonzero(solver.sibc_g > 0))


## 4. Run the FD impedance sweep

We compute the longitudinal impedance up to 3 GHz and also request a field monitor at 1 GHz.


In [ ]:
wake = WakeSolver(beta=1.0)
monitor = FieldMonitor(frequencies=[monitor_frequency])

f_num, Z_num = solver.impedance_sweep(
    frequencies=frequencies,
    wake=wake,
    field_monitor=monitor,
)

print("Frequency points:", len(f_num))
print("Impedance points:", len(Z_num))


## 5. Analytical resistive-wall impedance

We compare the numerical impedance to the analytical thick-wall round-pipe approximation.
We plot both the total impedance and the impedance per unit length.


In [ ]:
def analytic_rw_impedance_round_pipe(
    frequencies, radius, sigma, length, sign_imag=+1.0
):
    omega = 2.0 * np.pi * frequencies
    Rs = np.sqrt(omega * mu_0 / (2.0 * sigma))
    Zs = (1.0 + 1j * sign_imag) * Rs
    Z_per_length = Zs / (2.0 * np.pi * radius)
    return length * Z_per_length, Z_per_length


Z_ana, Z_ana_per_m = analytic_rw_impedance_round_pipe(
    f_num, radius_inner, sigma_wall, pipe_length, sign_imag=+1.0
)

Z_num_per_m = Z_num / pipe_length

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(f_num * 1e-9, np.real(Z_num_per_m), label="Re(Z_num/L)")
ax.plot(f_num * 1e-9, np.real(Z_ana_per_m), "--", label="Re(Z_analytic/L)")
ax.plot(f_num * 1e-9, np.imag(Z_num_per_m), label="Im(Z_num/L)")
ax.plot(f_num * 1e-9, np.imag(Z_ana_per_m), "--", label="Im(Z_analytic/L)")
ax.set_xlabel("f [GHz]")
ax.set_ylabel("Z_parallel / L [Ohm/m]")
ax.set_title("Round resistive-wall pipe: numerical vs analytical impedance per length")
ax.grid(True)
ax.legend()
plt.show()


## 6. Quantitative comparison

We summarize the relative error in the real part and the magnitude of the complex impedance.
The real part is usually the cleanest quantity for a first validation.


In [ ]:
mask = np.abs(Z_ana_per_m) > 0

rel_err_real = np.linalg.norm(
    np.real(Z_num_per_m[mask] - Z_ana_per_m[mask])
) / np.linalg.norm(np.real(Z_ana_per_m[mask]))
rel_err_abs = np.linalg.norm(
    np.abs(Z_num_per_m[mask] - Z_ana_per_m[mask])
) / np.linalg.norm(np.abs(Z_ana_per_m[mask]))

print(f"Relative error in Re(Z/L): {rel_err_real:.4e}")
print(f"Relative error in |Z/L|   : {rel_err_abs:.4e}")

sqrt_f = np.sqrt(f_num)
coef_real = np.polyfit(sqrt_f, np.real(Z_num_per_m), 1)
print("Approximate linear fit Re(Z/L) = a*sqrt(f) + b")
print(f"a = {coef_real[0]:.6e}, b = {coef_real[1]:.6e}")


## 7. Conductivity scaling check

For a good conductor we expect

\[
Z_\parallel \propto \frac{1}{\sqrt{\sigma}}.
\]

We therefore sweep a few conductivities and monitor the impedance at 1 GHz.


In [ ]:
sigmas = [1e5, 1e6, 1e7, 5.8e7]
results = []

for sigma in sigmas:
    s = make_fd_solver(sigma=sigma, verbose=0)
    f_tmp, Z_tmp = s.impedance_sweep(
        frequencies=np.array([monitor_frequency]),
        wake=WakeSolver(beta=1.0),
    )
    z_val = Z_tmp[0] / pipe_length
    results.append((sigma, z_val))
    print(
        f"sigma={sigma:.2e} S/m, "
        f"Re(Z/L)={np.real(z_val):.6e}, "
        f"Re(Z/L)*sqrt(sigma)={np.real(z_val) * np.sqrt(sigma):.6e}"
    )


## 8. Optional: inspect the field monitor at 1 GHz

Depending on your `FieldMonitor` implementation, you can access and visualize the stored complex field data here.
The exact accessor names may depend on your local WAKIS version.


In [ ]:
print(monitor)
# Example (adjust to your local API if needed):
# Ez_1GHz = monitor.get_field("Ez", frequency=monitor_frequency)
# print(Ez_1GHz.shape)